# 🧠 Module 12 · Day 2 (Sunday): Memory, Agent Design and LangGraph
**Designed by Engr Usman Hasan** · Bano Qabil Generative AI course

Yesterday we built an agent by hand: **LLM + tools + loop**. Today we make it a *real* single agent.

| Part | What we do | Slides |
|---|---|---|
| 0 | Setup + the brain (LangChain chat model) | Setup |
| 1 | Tools the LangChain way (`@tool`) | Level 1 |
| 2 | **LangChain `create_agent`**: yesterday's loop in one line | Level 1 |
| 3 | **Memory**: short-term (the conversation) + long-term (a diary file) | Level 2 |
| 4 | **Safety**: the step limit | Level 3 |
| 5 | **LangGraph**: draw the agent as a graph, run it step by step, ask a human before ordering | Level 4 |
| 6 | 🏆 Boss level: YOUR own agent | Boss |

## ⚙️ Part 0: Setup
**Colab:** your 🔑 Secret (`GROQ_API_KEY` or `GEMINI_API_KEY`) from Day 1 is still there. Turn **Notebook access** ON for this notebook too.

In [ ]:
# install LangChain (building blocks), LangGraph (the agent's flowchart) and the two model connectors
%pip install -q "langchain==1.5.2" "langgraph==1.2.24" "langchain-groq==1.2.1" "langchain-google-genai==4.5.3" "python-dotenv==1.2.4"

In [ ]:
# ===== STEP 0: load your FREE API key (Colab) =====
# 'os' stores settings (like API keys) in the environment
import os
# Colab Secrets = the 🔑 icon on the left
from google.colab import userdata

# choose your brain: "groq" or "gemini"
PROVIDER = "groq"
# the name of the secret we need
KEY_NAME = "GROQ_API_KEY" if PROVIDER == "groq" else "GEMINI_API_KEY"
# read the key from Colab Secrets into the environment
os.environ[KEY_NAME] = userdata.get(KEY_NAME)
# confirm (we never print the key itself!)
print(f"✅ {KEY_NAME} loaded. Provider: {PROVIDER}")

In [ ]:
# ===== THE BRAIN: a LangChain chat model (same as Modules 02-04) =====
# Groq's model, through LangChain
if PROVIDER == "groq":
    # the LangChain connector for Groq
    from langchain_groq import ChatGroq
    # temperature=0 -> no random answers
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, api_key=os.environ["GROQ_API_KEY"])
# Gemini's model, through LangChain
else:
    # the LangChain connector for Gemini
    from langchain_google_genai import ChatGoogleGenerativeAI
    # "gemini-flash-latest" always points to Google's newest Flash model
    llm = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0, api_key=os.environ["GEMINI_API_KEY"])

# quick test: one normal message in, one reply out
print(llm.invoke("Say salaam to the class in one line.").text)

## 🔧 Part 1: Tools, the LangChain way
Same farm, same 4 tools as yesterday. Only the wrapping changes: `@tool` + **type hints** (`product: str`) + a docstring.

In [ ]:
# ===== THE FARM'S DATA (same as yesterday) =====
# price of each product in rupees
PRICES = {"milk": 220, "yogurt": 250, "desi ghee": 2600}
# unit of each product
UNITS = {"milk": "litre", "yogurt": "kg", "desi ghee": "kg"}
# today's stock
STOCK = {"milk": 40, "yogurt": 12, "desi ghee": 3}
# the farm's rules
FARM_RULES = [
    # delivery time
    "Delivery time: every morning from 6 am to 9 am.",
    # delivery areas
    "Delivery areas: DHA, Clifton, Gulshan and PECHS only.",
    # delivery charge
    "Delivery charge: Rs 150 per order. Delivery is free when the bill is Rs 3000 or more.",
    # discount
    "Bulk discount: 5% off milk when a customer orders 20 litres or more.",
    # order deadline
    "Orders for tomorrow must be placed before 8 pm today.",
    # payment
    "Payment: cash on delivery or JazzCash.",
]

# ===== THE TOOLS, LangChain style (you met @tool in Module 04!) =====
# @tool turns a normal function into a tool. LangChain reads the NAME, the INPUTS (with their types)
# and the DOCSTRING, and sends them to the LLM. No tool menu or parser to write by hand any more.
from langchain_core.tools import tool


# tool 1: price
@tool
def get_price(product: str) -> str:
    """Price of ONE farm product (milk, yogurt or desi ghee)."""
    # clean the input ("  Milk " -> "milk")
    name = product.lower().strip()
    # unknown product -> say what we sell; known -> give the price
    return f"{name}: Rs {PRICES[name]} per {UNITS[name]}" if name in PRICES else f"We sell only: {', '.join(PRICES)}"


# tool 2: stock
@tool
def check_stock(product: str) -> str:
    """How much of ONE product is in stock today."""
    # clean the input
    name = product.lower().strip()
    # unknown product -> say what we sell; known -> give the stock
    return f"{name}: {STOCK[name]} {UNITS[name]} in stock" if name in STOCK else f"We sell only: {', '.join(STOCK)}"


# tool 3: calculator
@tool
def calculator(expression: str) -> str:
    """Do maths, e.g. '15 * 220' or '25 * 220 * 0.95'."""
    # only plain maths characters are allowed (safety)
    if any(ch not in "0123456789+-*/(). " for ch in expression):
        # refuse anything else
        return "Error: only numbers and + - * / ( ) are allowed."
    # calculate and round to 2 decimals
    return str(round(eval(expression), 2))


# tool 4: search the rules
@tool
def search_farm_rules(question: str) -> str:
    """Search the farm's rules: delivery time, areas, delivery charge, discounts, payment."""
    # the question's words longer than 3 letters
    words = [w.strip("?.,!") for w in question.lower().split() if len(w) > 3]
    # score each rule by how many of those words it contains
    scored = [(sum(w in rule.lower() for w in words), rule) for rule in FARM_RULES]
    # the 2 best matching rules
    best = [rule for score, rule in sorted(scored, reverse=True)[:2] if score > 0]
    # return them, or say nothing matched
    return "\n".join(best) if best else "No rule found."


# the agent's toolbox (a simple list now)
farm_tools = [get_price, check_stock, calculator, search_farm_rules]
# a @tool can still be run by hand with .invoke({...})
print(get_price.invoke({"product": "milk"}))
# what LangChain will show the LLM for one tool
print(calculator.name, "->", calculator.description, "->", calculator.args)

**Observe:** LangChain read the docstring as the **description** and the type hint as the **input format** (`args`).
Yesterday we wrote the tool menu and the parser ourselves. Today modern LLMs do **native tool calling**: they return a structured "please call this tool" message (you saw this in Module 04).

## 🤖 Part 2: LangChain's ready-made agent
`create_agent` gives us yesterday's whole loop (Think → Act → Observe → repeat) in **one line**.

In [ ]:
# ===== A HELPER TO SEE THE AGENT THINK =====
# print every message in the agent's conversation with an emoji
def show_steps(messages):
    # go through the messages in order
    for m in messages:
        # the customer's message
        if m.type == "human":
            # 🙋 = user
            print("🙋 USER:", m.text)
        # the LLM asked for tool(s)
        elif m.type == "ai" and m.tool_calls:
            # one line per tool call
            for call in m.tool_calls:
                # 🔧 = action (tool name + its inputs)
                print(f"🔧 ACTION: {call['name']}({call['args']})")
        # the LLM answered in words (no tool call) -> final answer
        elif m.type == "ai":
            # ✅ = answer
            print("✅ ANSWER:", m.text)
        # the result of a tool
        elif m.type == "tool":
            # 👀 = observation
            print("👀 OBSERVATION:", m.text)

In [ ]:
# ===== THE AGENT'S INSTRUCTIONS (system prompt) =====
# no Thought/Action format needed any more: the model's built-in tool calling does that part
SYSTEM_PROMPT = """You are the helpful agent of Marhaba Cattle & Dairy Farm in Karachi.
Use your tools for prices, stock, rules and maths. Never guess numbers.
Reply in the customer's language (English or Roman Urdu), short and friendly."""

# create_agent = LangChain's ready-made agent (Think -> Act -> Observe loop already built in)
from langchain.agents import create_agent

# brain + tools + instructions = an agent (yesterday's 40 lines, now 1 line!)
agent = create_agent(model=llm, tools=farm_tools, system_prompt=SYSTEM_PROMPT)

# ask yesterday's 3-step question
result = agent.invoke({"messages": [{"role": "user", "content": "Ali ko kal 15 litre doodh chahiye. Total bill kitna banega, aur kya itna stock hai?"}]})
# see every step the agent took
show_steps(result["messages"])

**What to notice:** the same steps as yesterday (🔧 Action → 👀 Observation → … → ✅ Answer), but we didn't write the loop, the menu or the parser.

> 💡 Fun fact: `create_agent` is **built on LangGraph** under the hood. Keep that in mind for Part 5.

## 🧠 Part 3: Memory
### 3A. The problem: an agent with no memory
Two separate questions. The second one needs the first.

In [ ]:
# question 1: introduce yourself
first = agent.invoke({"messages": [{"role": "user", "content": "Mera naam Ali hai aur main DHA mein rehta hoon."}]})
# print the reply
print("Reply 1:", first["messages"][-1].text)

# question 2: a NEW invoke, it starts from zero again
second = agent.invoke({"messages": [{"role": "user", "content": "Mera naam kya hai?"}]})
# does it remember?
print("Reply 2:", second["messages"][-1].text)

**Observe:** it forgot! Every `invoke` starts a fresh conversation. Like a shopkeeper who forgets you the moment you turn around.

### 3B. Short-term memory: a checkpointer + a thread id
A **checkpointer** saves the conversation after every step. A **thread_id** is the conversation's name (like a WhatsApp chat).
Same thread = same memory. New thread = fresh start.

In [ ]:
# InMemorySaver = a notebook in the computer's RAM that saves each conversation
from langgraph.checkpoint.memory import InMemorySaver

# the same agent as before, but now with memory
agent_with_memory = create_agent(model=llm, tools=farm_tools, system_prompt=SYSTEM_PROMPT,
                                 # save the conversation after every step
                                 checkpointer=InMemorySaver())

# the name of Ali's chat (like a WhatsApp chat id)
ali_chat = {"configurable": {"thread_id": "ali-chat"}}
# message 1 in Ali's chat
agent_with_memory.invoke({"messages": [{"role": "user", "content": "Mera naam Ali hai aur main DHA mein rehta hoon."}]}, ali_chat)
# message 2 in the SAME chat
reply = agent_with_memory.invoke({"messages": [{"role": "user", "content": "Mera naam kya hai, aur main kahan rehta hoon?"}]}, ali_chat)
# it should remember now
print("Same chat:", reply["messages"][-1].text)

# a DIFFERENT chat (Sara's) knows nothing about Ali
sara_chat = {"configurable": {"thread_id": "sara-chat"}}
# ask the same question in Sara's chat
reply = agent_with_memory.invoke({"messages": [{"role": "user", "content": "Mera naam kya hai?"}]}, sara_chat)
# it should NOT know
print("New chat:", reply["messages"][-1].text)

### 3C. Long-term memory: the agent's diary 📒
Short-term memory lives in RAM: restart the notebook and it's gone, and other chats can't see it.
**Long-term memory** is saved in a file (or a database). We give the agent two new tools: **write** in the diary and **read** the diary.

In [ ]:
# 'json' reads and writes the diary file
import json
# the diary file's name
NOTES_FILE = "customer_notes.json"


# read the whole diary (an empty diary if the file doesn't exist yet)
def load_notes():
    # if the file exists, read it
    if os.path.exists(NOTES_FILE):
        # open and read the JSON
        return json.load(open(NOTES_FILE, encoding="utf-8"))
    # no file yet -> empty diary
    return {}


# tool 5: write a note about a customer
@tool
def remember_customer(name: str, note: str) -> str:
    """Save a useful fact about a customer for the future (e.g. area, usual order, delivery time)."""
    # read the diary
    notes = load_notes()
    # add the note to this customer's list (names stored in small letters)
    notes.setdefault(name.lower(), []).append(note)
    # write the diary back to the file
    json.dump(notes, open(NOTES_FILE, "w", encoding="utf-8"), indent=2)
    # confirm
    return f"Saved for {name}: {note}"


# tool 6: read the notes about a customer
@tool
def recall_customer(name: str) -> str:
    """Read everything we saved earlier about a customer."""
    # this customer's notes (empty list if none)
    found = load_notes().get(name.lower(), [])
    # return them, or say there's nothing
    return "; ".join(found) if found else f"No notes about {name} yet."


# the instructions now tell the agent WHEN to use the diary
MEMORY_PROMPT = SYSTEM_PROMPT + """
When a customer tells you their name, first use recall_customer.
When they tell you something useful for next time (area, usual order, timing), save it with remember_customer."""

# an agent with 6 tools: farm tools + the 2 diary tools
diary_agent = create_agent(model=llm, tools=farm_tools + [remember_customer, recall_customer],
                           # the new instructions
                           system_prompt=MEMORY_PROMPT, checkpointer=InMemorySaver())
# chat 1 (Monday): Ali tells us something
r = diary_agent.invoke({"messages": [{"role": "user", "content": "Main Ali hoon. Mujhe har roz subah 7 baje 5 litre doodh chahiye hota hai, DHA Phase 6 mein."}]},
                       # chat 1 has its own thread id
                       {"configurable": {"thread_id": "monday"}})
# see the steps (look for remember_customer!)
show_steps(r["messages"])

In [ ]:
# chat 2 (Tuesday): a BRAND NEW thread, short-term memory is empty...
r = diary_agent.invoke({"messages": [{"role": "user", "content": "Main Ali hoon. Mera usual order kya hai aur uska bill kitna hoga?"}]},
                       # a different thread id
                       {"configurable": {"thread_id": "tuesday"}})
# ...but the diary remembers (look for recall_customer!)
show_steps(r["messages"])
# open the diary file and look inside
print("\n📒 Diary file:", load_notes())

**What to notice: three kinds of memory**
| Memory | Like… | In our code | Lasts |
|---|---|---|---|
| Short-term | remembering what you said a minute ago | `checkpointer` + `thread_id` | one conversation |
| Long-term | the shopkeeper's diary | `remember_customer` / `recall_customer` + a JSON file | forever |
| Knowledge | the farm's filing cabinet | `search_farm_rules` (RAG, Modules 08–09) | forever (read-only) |

## 🛡️ Part 4: Safety: the step limit
Yesterday we had `max_steps`. LangChain/LangGraph call it **`recursion_limit`**: the maximum number of steps before the agent is stopped.

In [ ]:
# GraphRecursionError is what we get when the limit is reached
from langgraph.errors import GraphRecursionError

# try the 3-step question with a VERY small limit
try:
    # 3 steps allowed (every "think" and every "tool run" counts as a step)
    agent.invoke({"messages": [{"role": "user", "content": "15 litre doodh ka bill aur stock batao."}]}, {"recursion_limit": 3})
# the seatbelt worked
except GraphRecursionError:
    # explain what happened
    print("⛔ Stopped by recursion_limit. The agent needed more steps than we allowed.")

## 🕸️ Part 5: LangGraph: where LangChain ends and LangGraph starts
| | **LangChain** | **LangGraph** |
|---|---|---|
| Think of it as | 🧱 **building blocks**: models, prompts, `@tool`, retrievers, loaders | 🗺️ **the map** that connects the blocks: steps, loops, branches |
| Shape | mostly a straight line: A → B → C | a **graph**: arrows can go back (**loops**) and split (**if/else**) |
| Memory | you pass messages yourself | built-in **state** + **checkpointer** (save, pause, resume) |
| Human check | hard | built-in **interrupt**: pause, ask a human, continue |
| Best for | chains, RAG, quick agents (`create_agent`) | custom agents you fully control and can debug step by step |

LangGraph is made by the LangChain team. `create_agent` is a **ready-made LangGraph graph**. Now we build that graph **ourselves** to see inside.

**Three words:** **State** (the shared notebook every step reads and writes) · **Node** (one step = a Python function) · **Edge** (an arrow: which step comes next).

In [ ]:
# ===== 5A. THE STATE: the shared notebook every node reads and writes =====
# TypedDict = a dictionary with fixed keys; Annotated = a type with an extra note
from typing import Annotated, TypedDict
# add_messages = "when a node returns new messages, ADD them to the list (don't replace it)"
from langgraph.graph.message import add_messages


# our state has one key: the list of messages (the conversation)
class FarmState(TypedDict):
    # the conversation; new messages get added to the end
    messages: Annotated[list, add_messages]

In [ ]:
# ===== 5B. THE NODES: each node is one step (a Python function) =====
# SystemMessage = the instructions message
from langchain_core.messages import SystemMessage
# ToolNode = a ready-made node that runs whatever tools the LLM asked for
from langgraph.prebuilt import ToolNode

# give the LLM the list of tools it may ask for
llm_with_tools = llm.bind_tools(farm_tools)


# NODE 1, "agent": the brain thinks (answers, or asks for a tool)
def agent_node(state: FarmState):
    # instructions + the conversation so far -> the LLM
    reply = llm_with_tools.invoke([SystemMessage(SYSTEM_PROMPT)] + state["messages"])
    # return the reply; add_messages appends it to the state
    return {"messages": [reply]}


# NODE 2, "tools": runs the tools the LLM asked for and adds their results to the state
tools_node = ToolNode(farm_tools)

In [ ]:
# ===== 5C. THE EDGES: draw the map =====
# StateGraph = an empty map; START and END = the entry and exit points
from langgraph.graph import StateGraph, START, END
# tools_condition = "did the LLM ask for a tool? then go to 'tools', else go to END"
from langgraph.prebuilt import tools_condition

# an empty map that uses our state
builder = StateGraph(FarmState)
# place node 1 on the map
builder.add_node("agent", agent_node)
# place node 2 on the map
builder.add_node("tools", tools_node)
# arrow: start -> agent
builder.add_edge(START, "agent")
# IF/ELSE arrow: agent -> tools (a tool was requested) OR agent -> END (final answer)
builder.add_conditional_edges("agent", tools_condition)
# arrow back: tools -> agent  (THIS is the loop!)
builder.add_edge("tools", "agent")
# compile = turn the map into a runnable agent (with memory)
farm_graph = builder.compile(checkpointer=InMemorySaver())

# draw the map as a picture (needs internet); if that fails, print it as text
try:
    # Image + display show pictures inside the notebook
    from IPython.display import Image, display
    # draw it
    display(Image(farm_graph.get_graph().draw_mermaid_png()))
# no internet / drawing service busy
except Exception:
    # the same map as text
    print(farm_graph.get_graph().draw_mermaid())

**Look at the picture:** `start → agent → tools → agent → … → end`. The arrow from **tools back to agent** is the loop you wrote by hand yesterday.
The dotted arrows are the **if/else** decision.

In [ ]:
# ===== 5D. RUN IT AND WATCH THE MAP, step by step =====
# a new chat for this test
config = {"configurable": {"thread_id": "graph-test"}}
# the question
question = {"messages": [{"role": "user", "content": "25 litre doodh ka bill discount ke saath kitna hoga?"}]}
# stream_mode="updates" -> give us each node's output the moment it finishes
for update in farm_graph.stream(question, config, stream_mode="updates"):
    # each update = {node name: what that node returned}
    for node_name, output in update.items():
        # which box on the map just ran
        print(f"\n📍 NODE: {node_name}")
        # what it added to the state
        show_steps(output["messages"])

### 5E. 🛑 Human in the loop: ask the manager before placing an order
Reading prices is safe. **Placing an order** changes the real world (stock goes down, a rider goes out).
A good agent **pauses and asks a human** before risky actions. In LangGraph this is one function: `interrupt()`.

In [ ]:
# interrupt = pause the graph and wait for a human; Command = how we send the human's answer back
from langgraph.types import interrupt, Command


# tool 7: place an order (RISKY -> needs a human's OK)
@tool
def place_order(customer: str, product: str, quantity: float) -> str:
    """Place a delivery order for a customer. Use only after the customer clearly asks to order."""
    # PAUSE here and show the manager what the agent wants to do
    decision = interrupt(f"Approve order: {quantity} {product} for {customer}? (yes/no)")
    # the manager said no -> don't order
    if str(decision).lower().strip() != "yes":
        # tell the agent it was cancelled
        return "The farm manager did NOT approve this order. Tell the customer politely."
    # not enough stock -> don't order
    if STOCK.get(product.lower(), 0) < quantity:
        # tell the agent
        return f"Not enough {product} in stock."
    # take the quantity out of the stock
    STOCK[product.lower()] -= quantity
    # confirm
    return f"Order placed: {quantity} {product} for {customer}. Stock left: {STOCK[product.lower()]}."


# the toolbox now has the risky tool too
order_tools = farm_tools + [place_order]
# the brain must know about the new tool
llm_with_tools = llm.bind_tools(order_tools)
# rebuild the same map, now with the order tool in the tools node
builder = StateGraph(FarmState)
# brain node (agent_node uses the new llm_with_tools)
builder.add_node("agent", agent_node)
# tools node with 5 tools
builder.add_node("tools", ToolNode(order_tools))
# start -> agent
builder.add_edge(START, "agent")
# agent -> tools or END
builder.add_conditional_edges("agent", tools_condition)
# tools -> agent (the loop)
builder.add_edge("tools", "agent")
# compile with memory (a checkpointer is REQUIRED for pausing)
order_graph = builder.compile(checkpointer=InMemorySaver())
# ready
print("Order agent ready with tools:", [t.name for t in order_tools])

In [ ]:
# Ali asks the agent to place an order
config = {"configurable": {"thread_id": "ali-order"}}
# run until the graph pauses
result = order_graph.invoke({"messages": [{"role": "user", "content": "Main Ali hoon. Kal ke liye 10 litre doodh order kar dein."}]}, config)

# did it pause? (the pause shows up under "__interrupt__")
if "__interrupt__" in result:
    # the question the agent is asking the manager
    print("🛑 AGENT PAUSED:", result["__interrupt__"][0].value)
    # YOU are the farm manager: type yes or no
    decision = input("Manager, type yes or no: ")
    # resume the graph from exactly where it paused, with your decision
    result = order_graph.invoke(Command(resume=decision), config)

# see the whole story
show_steps(result["messages"])
# the stock after the order
print("\nMilk stock now:", STOCK["milk"], "litres")

**What to notice**
- The graph **stopped in the middle**, waited for a human, then **continued from the same spot**. That's the checkpointer at work.
- Try again and type `no`. The agent should tell Ali politely.
- This pause-and-resume is the main thing **LangGraph** gives us that a plain LangChain chain can't.

## 🏆 Part 6: Boss level: YOUR own single agent
Use your project idea from Saturday's homework. Fill in the 4 parts of the **agent blueprint**:
1. **Brain:** `llm` (already made)
2. **Instructions:** who is the agent, what it must never do
3. **Tools:** 2–3 functions with clear docstrings
4. **Memory + safety:** checkpointer, `recursion_limit`, and `interrupt()` for risky actions

The example below is a **Bano Qabil class helper**. Change it to YOUR idea.

In [ ]:
# ===== ✍️ YOUR AGENT (example: a Bano Qabil class helper; change everything!) =====
# 1. the data your tools use
TIMETABLE = {"saturday": "Generative AI, 10 am - 12 pm", "sunday": "Generative AI, 10 am - 12 pm"}


# 2. tool A
@tool
def class_time(day: str) -> str:
    """Which class runs on a day of the week, and its time."""
    # look up the day (small letters); unknown day -> no class
    return TIMETABLE.get(day.lower().strip(), f"No class on {day}.")


# 2. tool B
@tool
def days_until_exam(today_day_number: int) -> str:
    """How many days are left until the exam (exam is on day 30). Input: today's day number of the month."""
    # simple maths
    return f"{30 - today_day_number} days left until the exam."


# 3. instructions
MY_PROMPT = "You are a friendly helper for Bano Qabil students. Use tools for timings. Never invent class times."
# 4. brain + tools + instructions + memory = your agent (create_agent builds the LangGraph graph for you)
my_agent = create_agent(model=llm, tools=[class_time, days_until_exam], system_prompt=MY_PROMPT, checkpointer=InMemorySaver())

# test it (with a safety limit of 10 steps)
r = my_agent.invoke({"messages": [{"role": "user", "content": "Saturday ko kaunsi class hai? Aur aaj 18 tareekh hai, exam mein kitne din hain?"}]},
                    # a chat id + the safety limit
                    {"configurable": {"thread_id": "me"}, "recursion_limit": 10})
# see it think
show_steps(r["messages"])

### Present your agent (2 minutes)
1. What problem does it solve? 2. Its tools (and their one-line descriptions). 3. What does it remember? 4. Which action needs a human's OK? 5. Show one run.

## 🎉 Module 12 complete: Shabash!
| You learned | In one line |
|---|---|
| Agent | LLM + tools + loop; it **decides** the steps |
| ReAct | Thought → Action → Observation → … → Final Answer |
| Tools | Python functions + **clear descriptions** |
| Memory | short-term (thread), long-term (diary), knowledge (RAG) |
| Single-agent design | brain · instructions · tools · memory · loop · safety |
| LangChain vs LangGraph | blocks vs the map: state, loops, if/else, pause for a human |
---
© Module 12 course material designed by **Engr Usman Hasan**.